# 02 — LLM Generation Protocol & Prompts  [R1-5]
The exact system prompt and data profile sent to each model, the raw responses, and the frozen preprocessing code evaluated in the paper. Generation: OpenRouter, 28 Jun 2026, temperature 0.0, models `deepseek/deepseek-chat-v3.1`, `anthropic/claude-sonnet-4.6`, `openai/gpt-5.4`.

In [1]:
import json, inspect, warnings, os, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedKFold, cross_val_score, GridSearchCV, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import (StandardScaler, RobustScaler, PowerTransformer, MinMaxScaler, QuantileTransformer)
from sklearn.metrics import (accuracy_score, matthews_corrcoef, roc_auc_score, recall_score, brier_score_loss)
from sklearn.calibration import calibration_curve
from scipy import stats
from lightgbm import LGBMClassifier
os.makedirs("results", exist_ok=True)
RS = 42
TEAL,ROSE,ORANGE,SAND,RED,LAV,SLATE,INK = ('#4A90A4','#C4887D','#E67E22','#C4A882','#C0392B','#8E7CC3','#7A8B9A','#2C3E4F')
plt.rcParams.update({'font.family':'DejaVu Sans','axes.facecolor':'#FAFBFC','figure.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.color':'#E4E9F0',
    'axes.titleweight':'bold','axes.axisbelow':True,'savefig.dpi':200,'savefig.bbox':'tight'})
NAMES = ['Manual Expert','DeepSeek','Claude Sonnet 4.6','GPT-5.4','CAAFE']
SHORT = ['Manual','DeepSeek','Claude 4.6','GPT-5.4','CAAFE']
COLS  = [TEAL,ROSE,RED,ORANGE,LAV]
CONT  = ['AGE','ANNUALCONTRIBUTION','ANNUALCLAIMAMOUNT','UNITSTOTAL']
DATA  = "Final Prepared Dataset - Diabetes and Hypertension Data.xlsx"

def load_raw():
    df = pd.read_excel(DATA)
    n_raw = len(df); df = df.drop_duplicates().reset_index(drop=True); n_dup = n_raw - len(df)
    df["ADHERENCE"] = (df["ADHERENCE"]=="ADHERENT").astype(int)
    for c in df.columns:
        if df[c].dtype=="bool": df[c]=df[c].astype(int)
    return df, n_raw, n_dup

def load_and_split():
    df,_,_ = load_raw()
    y=df["ADHERENCE"]; X=df.drop(columns=["ADHERENCE"])
    return train_test_split(X,y,test_size=0.2,stratify=y,random_state=RS)

def _manual(Xp):
    Xp=Xp.copy()
    Xp["LOG_CLAIM"]=np.log1p(Xp["ANNUALCLAIMAMOUNT"]); Xp["LOG_UNITS"]=np.log1p(Xp["UNITSTOTAL"])
    Xp["LOG_CONTRIB"]=np.log1p(Xp["ANNUALCONTRIBUTION"])
    Xp["CLAIM_PER_UNIT"]=np.log1p(Xp["ANNUALCLAIMAMOUNT"]/(Xp["UNITSTOTAL"]+1))
    Xp["CLAIM_RATIO"]=np.log1p(Xp["ANNUALCLAIMAMOUNT"]/(Xp["ANNUALCONTRIBUTION"]+1))
    Xp["UNITS_PER_AGE"]=Xp["UNITSTOTAL"]/(Xp["AGE"]+1); return Xp

_NS={"np":np,"pd":pd,"StandardScaler":StandardScaler,"RobustScaler":RobustScaler,
     "PowerTransformer":PowerTransformer,"MinMaxScaler":MinMaxScaler,"QuantileTransformer":QuantileTransformer}
def _load(p):
    ns=dict(_NS); exec(open(p,encoding="utf-8").read(),ns); return ns["preprocess"]
FROZEN={"DeepSeek":_load("llm_raw/DeepSeek_preprocess.py"),
        "Claude Sonnet 4.6":_load("llm_raw/Claude_Sonnet_4.6_preprocess.py"),
        "GPT-5.4":_load("llm_raw/GPT-5.4_preprocess.py")}
def _caafe(frame):
    ns={"df":frame.copy(),"np":np,"pd":pd}; exec(open("llm_raw/CAAFE_code.py",encoding="utf-8").read(),ns)
    return ns["df"].select_dtypes(include=[np.number]).fillna(0)
def transform_pair(name,x_fit,x_eval):
    if name=="Manual Expert": return _manual(x_fit),_manual(x_eval)
    if name=="CAAFE":
        tr=_caafe(x_fit); te=_caafe(x_eval).reindex(columns=tr.columns,fill_value=0); return tr,te
    tr,te=FROZEN[name](x_fit.copy(),x_eval.copy())
    if not isinstance(tr,pd.DataFrame):
        cols=(list(x_fit.columns)+["CONTRIBUTION_CLAIM_RATIO","UNITS_PER_CLAIM"]) if (name=="DeepSeek" and tr.shape[1]==x_fit.shape[1]+2) else [f"f{i}" for i in range(tr.shape[1])]
        tr=pd.DataFrame(tr,index=x_fit.index,columns=cols)
    if not isinstance(te,pd.DataFrame): te=pd.DataFrame(te,index=x_eval.index,columns=tr.columns)
    return tr,te.reindex(columns=tr.columns,fill_value=0)
def san(df): return pd.DataFrame(df).replace([np.inf,-np.inf],np.nan).fillna(0.0)
def nb_se(d,k=10):
    d=np.asarray(d); n=d.size; return np.sqrt(d.var(ddof=1)*(1.0/n+(1.0/k)/(1-1.0/k)))
def tost90(d,k=10):
    d=np.asarray(d); m=d.mean(); se=nb_se(d,k); c=stats.t.ppf(0.95,d.size-1); return m,se,[m-c*se,m+c*se]
def cv_metrics(name,X,y,drop_units=False,k=10,repeats=1):
    accs,mccs,aucs=[],[],[]
    for r in range(repeats):
        for tr,va in StratifiedKFold(k,shuffle=True,random_state=(RS if repeats==1 else 2026+r)).split(X,y):
            ftr,fva=transform_pair(name,X.iloc[tr],X.iloc[va])
            if drop_units:
                uc=[c for c in ftr.columns if 'UNIT' in c.upper()]
                ftr=ftr.drop(columns=uc); fva=fva.drop(columns=[c for c in uc if c in fva.columns])
            ftr,fva=san(ftr),san(fva)
            m=LGBMClassifier(random_state=RS,verbose=-1,n_jobs=1).fit(ftr,y.iloc[tr])
            p=m.predict(fva); pr=m.predict_proba(fva)[:,1]
            accs.append(accuracy_score(y.iloc[va],p)); mccs.append(matthews_corrcoef(y.iloc[va],p)); aucs.append(roc_auc_score(y.iloc[va],pr))
    return np.array(accs),np.array(mccs),np.array(aucs)
x_tr,x_ho,y_tr,y_ho = load_and_split()
print("Setup complete |", f"{len(x_tr)+len(x_ho):,} records, {x_tr.shape[1]} predictors")

Setup complete | 24,071 records, 11 predictors


### System prompt (verbatim)

In [2]:
SYSTEM_PROMPT = ("You are a senior data scientist. Analyze this medical-claims dataset profile "
"and generate the best preprocessing for LightGBM classification (skewness fixes, feature "
"engineering such as ratios/interactions, and scaling). Return ONE function "
"def preprocess(X_train, X_test=None): that fits any stateful transformer on X_train only and "
"applies the learned state to both. Output ONLY Python in a ```python block.")
print(SYSTEM_PROMPT)

You are a senior data scientist. Analyze this medical-claims dataset profile and generate the best preprocessing for LightGBM classification (skewness fixes, feature engineering such as ratios/interactions, and scaling). Return ONE function def preprocess(X_train, X_test=None): that fits any stateful transformer on X_train only and applies the learned state to both. Output ONLY Python in a ```python block.


### Raw model responses (excerpt)

In [3]:
for f in ['DeepSeek_response.txt','Claude_Sonnet_4.6_response.txt','GPT-5.4_response.txt']:
    txt=open('llm_raw/'+f,encoding='utf-8').read()
    print('='*70,'\n',f,f'({len(txt)} chars)\n','='*70); print(txt[:600],'\n...')

 DeepSeek_response.txt (2041 chars)
```python
import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer, StandardScaler

def preprocess(X_train, X_test=None):
    """
    Preprocess medical claims data with skewness correction, feature engineering,
    and scaling. Fits transformers on X_train only.
    """
    # Create copies to avoid modifying original data
    X_train_processed = X_train.copy()
    if X_test is not None:
        X_test_processed = X_test.copy()
    
    # Feature engineering: Create ratio features
    X_train_processed['CONTRIBUTION_CLAIM_RATIO'] = X_train_processed['ANNUALCONT 
...
 Claude_Sonnet_4.6_response.txt (3803 chars)
```python
def preprocess(X_train, X_test=None):
    import numpy as np
    import pandas as pd
    from sklearn.preprocessing import RobustScaler

    # ── helpers ──────────────────────────────────────────────────────────────
    CONTINUOUS = ['AGE', 'ANNUALCONTRIBUTION', 'ANNUALCLAIMAMOUNT', 'UNITSTOTAL']
    

### Frozen preprocessing code evaluated in the paper

In [4]:
for f in ['DeepSeek_preprocess.py','Claude_Sonnet_4.6_preprocess.py','GPT-5.4_preprocess.py','CAAFE_code.py']:
    print('\n#','='*60,f,'='*(60-len(f)),'\n'); print(open('llm_raw/'+f,encoding='utf-8').read())


# ============================================================ DeepSeek_preprocess.py ====================================== 

import numpy as np
import pandas as pd
from sklearn.preprocessing import PowerTransformer, StandardScaler

def preprocess(X_train, X_test=None):
    """
    Preprocess medical claims data with skewness correction, feature engineering,
    and scaling. Fits transformers on X_train only.
    """
    # Create copies to avoid modifying original data
    X_train_processed = X_train.copy()
    if X_test is not None:
        X_test_processed = X_test.copy()
    
    # Feature engineering: Create ratio features
    X_train_processed['CONTRIBUTION_CLAIM_RATIO'] = X_train_processed['ANNUALCONTRIBUTION'] / (X_train_processed['ANNUALCLAIMAMOUNT'] + 1)
    X_train_processed['UNITS_PER_CLAIM'] = X_train_processed['UNITSTOTAL'] / (X_train_processed['ANNUALCLAIMAMOUNT'] + 1)
    
    if X_test is not None:
        X_test_processed['CONTRIBUTION_CLAIM_RATIO'] = X_test_processe

### Target-correlation sensitivity [R1-6] (archived; requires API to reproduce)

In [5]:
r16=pd.DataFrame([['DeepSeek',0.8194,0.8190,-0.0004],['Claude Sonnet 4.6',0.8186,0.8201,0.0015],
    ['GPT-5.4',0.8238,0.8221,-0.0017]],columns=['Model','With target-corr','Without target-corr','Delta'])
r16.to_excel('results/02_target_corr_sensitivity.xlsx',index=False); display(r16)
print('All |Delta| < 0.002 -> target correlations did not drive pipeline quality.')

,Model,With target-corr,Without target-corr,Delta
0,DeepSeek,0.8194,0.8190,-0.0004
1,Claude Sonnet 4.6,0.8186,0.8201,0.0015
2,GPT-5.4,0.8238,0.8221,-0.0017


All |Delta| < 0.002 -> target correlations did not drive pipeline quality.
